# 🩻 MedKnow — Teaching Medical AI When Not to Know

**One-click Colab demo for the [MedKnow](https://github.com/ojdanajakir848-a11y/medknow) benchmark.**

What you get in ~5 minutes on a free CPU/GPU runtime:
1. **Inference demo** — load the pretrained ResNet-18, run MC Dropout uncertainty on sample chest X-rays
2. **The money plot** — referral curves (internal vs RSNA vs NIH) showing that uncertainty-driven referral works in-domain but fails under domain shift
3. **Optional** — full paper reproduction (train from scratch on the Kermany dataset)

> ⚠️ Research / education only. Not a medical device.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ojdanajakir848-a11y/medknow/blob/main/notebooks/medknow_colab.ipynb)


## 0. Before you run

Two one-time edits in the cell below (search for `ojdanajakir848-a11y`):
- the **git clone URL** (replace with your GitHub username), and
- the **model weights URL** (after you publish the Hugging Face Space / a GitHub Release — either one works).


In [ ]:
# ==== EDIT THESE (replace ojdanajakir848-a11y) ====
GITHUB_USER   = "ojdanajakir848-a11y"          # your GitHub username
REPO_URL      = f"https://github.com/{GITHUB_USER}/medknow.git"
# Model weights: publish EITHER the HF Space (recommended) OR a GitHub Release.
HF_SPACE_URL  = f"https://huggingface.co/spaces/{GITHUB_USER}/medknow-pneumonia-xray/resolve/main/model.pth"
GH_RELEASE_URL= f"https://github.com/{GITHUB_USER}/medknow/releases/download/v1.0.0/model.pth"
# =========================================

import os
import subprocess
import urllib.request

if not os.path.isdir("medknow"):
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir("medknow")
print("Working in:", os.getcwd())

def download_model(dest="model.pth"):
    if os.path.exists(dest):
        print(f"{dest} already present, skipping download.")
        return
    candidates = [HF_SPACE_URL, GH_RELEASE_URL]
    for url in candidates:
        if "ojdanajakir848-a11y" in url:
            continue
        print(f"Trying {url} ...")
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"Downloaded {dest} ({os.path.getsize(dest)/1e6:.1f} MB)")
            return
        except Exception as e:  # noqa: BLE001 - download retry fallback
            print(f"  failed: {e}")
    raise RuntimeError("Could not download model weights. Publish the HF Space or a GitHub Release, then re-run this cell.")

download_model()


## 1. Install

Installs the `medknow` package (editable) plus the demo extras. On a free Colab
runtime this takes ~1-2 minutes.


In [ ]:
!pip install -q -e ".[dev]" grad-cam
!python -c "from medknow.models.factory import create_resnet18; print('medknow imports OK')"


## 2. Load the model

Loads the seed_42 checkpoint (`outputs/pneumonia_model.pth` in the repo).
The architecture matches the paper exactly: ResNet-18, frozen backbone,
layer4 + dropout head (p=0.3), trained on 5,856 chest X-rays.


In [ ]:
import os
import sys

sys.path.insert(0, ".")
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision import transforms

from config import DEVICE
from models.model_factory import enable_dropout, load_trained_model

model = load_trained_model(name="resnet18", num_classes=2, model_path="model.pth", device=DEVICE)
print(f"Model loaded on {DEVICE}")

# temperature scaling constant (T=1.67, fit on internal validation)
temperature = 1.67
try:
    with open("temperature.txt") as f:
        temperature = float(f.read().strip())
except FileNotFoundError:
    print("temperature.txt not found — using T=1.67 from the manuscript.")
print(f"Temperature T = {temperature:.3f}")

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


## 3. Inference + MC Dropout uncertainty on sample images

For each sample chest X-ray we run **30 stochastic forward passes** (dropout on)
and report the mean pneumonia probability **and** its standard deviation —
the model's uncertainty. High std ⇒ the model is not sure ⇒ refer to a human.


In [ ]:
import glob

import torch.nn.functional as F


def mc_inference(path, n_samples=30):
    img = Image.open(path).convert("RGB")
    x = tf(img).unsqueeze(0).to(DEVICE)
    model.eval(); enable_dropout(model)
    probs = []
    with torch.no_grad():
        for _ in range(n_samples):
            logits = model(x) / temperature
            probs.append(F.softmax(logits, dim=1)[0, 1].item())
    p = float(np.mean(probs)); s = float(np.std(probs))
    verdict = "uncertain → refer to human" if s > 0.05 else "confident"
    return img, p, s, verdict

example_paths = sorted(glob.glob("examples/*.jpeg"))[:4]
fig, axes = plt.subplots(1, len(example_paths), figsize=(4 * len(example_paths), 4))
if len(example_paths) == 1:
    axes = [axes]
for ax, path in zip(axes, example_paths):
    img, p, s, verdict = mc_inference(path)
    ax.imshow(img, cmap="gray"); ax.axis("off")
    ax.set_title(f"P(pneu)={p:.1%}\nstd={s:.3f} ({verdict})", fontsize=10)
plt.tight_layout(); plt.show()

print("Sample chest X-rays from the repo's examples/ folder.")


## 4. The money plot: referral curves under domain shift

This is the core finding of the paper. We route the *most uncertain* cases to a
human reader at each referral rate and measure the **error rate on the retained set**:

- **Internal test** — uncertainty-driven referral *works*: error drops from 4.0% to 0
  at ~30% referral (MC Dropout / MSP), far below random referral.
- **RSNA / NIH (external)** — the same signal becomes *no better than random*.

**A model can be confident and wrong under domain shift.** Confidence-based
medical-AI triage cannot be trusted on internal validation alone.


In [ ]:
import json

import matplotlib.pyplot as plt

with open("results/tables/referral_methods_summary.json") as f:
    data = json.load(f)

methods = [("Random", "random", True), ("MSP", "msp", False),
           ("MC Dropout", "mc_dropout", False), ("Ensemble", "ensemble", False)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, cohort in zip(axes, ["internal_test", "rsna", "nih"]):
    d = data[cohort]
    rates = d["rates"]
    for label, key, is_random in methods:
        m = d[key]
        y = m["retained_error_mean"] if is_random else m["retained_error"]
        ax.plot(rates, y, marker="o", markersize=4, label=label, linewidth=1.8)
    ax.set_title({"internal_test": "Internal test (in-domain)",
                  "rsna": "RSNA (external)", "nih": "NIH (external)"}[cohort], fontsize=12)
    ax.set_xlabel("Referral rate")
    ax.set_ylim(0, 0.45); ax.grid(alpha=0.3)
axes[0].set_ylabel("Retained-set error rate")
axes[0].legend(fontsize=9)
plt.suptitle("Uncertainty-driven referral: works in-domain, fails under domain shift", fontsize=13)
plt.tight_layout(); plt.show()


## 5. Optional: full reproduction (train from scratch)

The complete paper pipeline (data → train 3 seeds → uncertainty → calibration →
referral → external validation → figures) is documented in the README section
`*"Reproduce the experiments"*`. To run it on Colab you need the Kermany dataset
(~1.3 GB):

```bash
# 1) Get the dataset (Kaggle: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)
#    or any Kermany mirror, then place it at data/chest_xray/
# 2) Patient-level split + train (free T4 GPU: ~15-30 min)
python scripts/medknow_split_patient.py --seed 42
python scripts/medknow_train.py --config configs/baseline.yaml --seed 42
# 3) Evaluate + uncertainty + referral + external validation
python scripts/medknow_evaluate.py --weights checkpoints/seed_42.pth
python scripts/medknow_run_referral.py --weights checkpoints/seed_42.pth
python scripts/medknow_evaluate_external.py --weights checkpoints/seed_42.pth
```

All manuscript numbers are verified reproducible — see
`results/tables/manuscript_verification.md`.


## 6. Citation & star

If this project is useful, please ⭐ the [repository](https://github.com/ojdanajakir848-a11y/medknow)
and cite it:

```bibtex
@software{medknow2026,
  author = {MedKnow authors},
  title  = {MedKnow: Teaching Medical AI When Not to Know},
  year   = {2026},
  url    = {https://github.com/ojdanajakir848-a11y/medknow}
}
```

- Paper manuscript: `paper/output/doc/manuscript.md`
- Interactive demo: Hugging Face Space
- License: MIT
